In [1]:
import os
import glob
import polars as pl

# ==========================================
# 1. CONFIGURATION PATHS & CONSTANTS
# ==========================================
DATA_DIR = r"D:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds"
TARGET_MAP = "ancient"
MATCHES_TO_PROCESS = 10

def find_ancient_bounds():
    print(f"Searching for {MATCHES_TO_PROCESS} matches on map '{TARGET_MAP}'...\n")
    
    # Initialize global boundary tracking variables
    global_min_x = float('inf')
    global_max_x = float('-inf')
    global_min_y = float('inf')
    global_max_y = float('-inf')
    
    matches_processed = 0
    
    # ==========================================
    # 2. FILE DISCOVERY & PARSING
    # ==========================================
    # Find all header files recursively, ignoring exact extensions
    search_pattern = os.path.join(DATA_DIR, "**", "header*")
    header_files = glob.glob(search_pattern, recursive=True)
    
    for header_path in header_files:
        if not os.path.isfile(header_path):
            continue
            
        match_dir = os.path.dirname(header_path)
        match_id = os.path.basename(match_dir)
        
        # Attempt to read the header file to verify the map
        try:
            df_header = pl.read_parquet(header_path)
            raw_map_name = str(df_header["map_name"][0]).lower()
            map_name = raw_map_name.replace("de_", "")  # Normalize 'de_ancient' to 'ancient'
        except Exception:
            # Skip corrupted/invalid files silently
            continue
            
        # ==========================================
        # 3. WEAPON FIRE PARSING & BOUNDARY EXTRACTION
        # ==========================================
        if map_name == TARGET_MAP:
            wf_pattern = os.path.join(match_dir, "weapon_fire*")
            wf_files = glob.glob(wf_pattern)
            
            if not wf_files:
                continue
                
            wf_path = wf_files[0]
            
            try:
                df_wf = pl.read_parquet(wf_path)
                
                # Verify columns exist
                if "player_x_pos" in df_wf.columns and "player_y_pos" in df_wf.columns:
                    # Find min/max for this specific match
                    local_min_x = df_wf["player_x_pos"].min()
                    local_max_x = df_wf["player_x_pos"].max()
                    local_min_y = df_wf["player_y_pos"].min()
                    local_max_y = df_wf["player_y_pos"].max()
                    
                    # Update the global map bounds
                    global_min_x = min(global_min_x, local_min_x)
                    global_max_x = max(global_max_x, local_max_x)
                    global_min_y = min(global_min_y, local_min_y)
                    global_max_y = max(global_max_y, local_max_y)
                    
                    matches_processed += 1
                    print(f"[{matches_processed}/{MATCHES_TO_PROCESS}] Processed bounds for match: {match_id}")
                    
                    # Stop once we hit our target sample size
                    if matches_processed >= MATCHES_TO_PROCESS:
                        break
            except Exception as e:
                print(f"Error reading weapon_fire for match {match_id}: {e}")
                continue

    # ==========================================
    # 4. RESULTS DISPLAY
    # ==========================================
    if matches_processed > 0:
        print("\n" + "="*50)
        print(f" EMPIRICAL MAP BOUNDS FOR: {TARGET_MAP.upper()}")
        print(f" Matches analyzed: {matches_processed}")
        print("="*50)
        print(f" x_min : {global_min_x:>10.2f}")
        print(f" x_max : {global_max_x:>10.2f}")
        print(f" y_min : {global_min_y:>10.2f}")
        print(f" y_max : {global_max_y:>10.2f}")
        print("="*50)
        
        # Print dictionary format for easy copy-pasting into your main script
        print("\nDictionary Output format:")
        print(f"MAP_BOUNDS_Ancient = {{'x_min': {global_min_x}, 'x_max': {global_max_x}, 'y_min': {global_min_y}, 'y_max': {global_max_y}}}")
    else:
        print(f"No valid matches found for map '{TARGET_MAP}'.")

if __name__ == "__main__":
    find_ancient_bounds()

Searching for 10 matches on map 'ancient'...

[1/10] Processed bounds for match: 095ec9ef-d502-413f-8f91-ff14304f461a
[2/10] Processed bounds for match: 0cc3d8b7-a337-45df-ac2d-0f342611ef60
[3/10] Processed bounds for match: 0d481617-c55f-4bb9-82bc-c729eb10b266
[4/10] Processed bounds for match: 0dcd5305-5ea5-46fb-b91f-4d43f5ea9b14
[5/10] Processed bounds for match: 0e6d5730-cb48-406f-a712-1edd70b56891
[6/10] Processed bounds for match: 15b419cb-69d4-4627-ad5b-40d81ad7b75a
[7/10] Processed bounds for match: 1e5341d1-b28f-49b0-84bc-0649d3ed60f8
[8/10] Processed bounds for match: 272cf9cd-2e81-44b5-9c95-a66dc5cbbead
[9/10] Processed bounds for match: 29ec627c-7c60-4083-ab36-5097428461aa
[10/10] Processed bounds for match: 2dbca0dc-f283-4f8f-9e24-1971978a9cbd

 EMPIRICAL MAP BOUNDS FOR: ANCIENT
 Matches analyzed: 10
 x_min :   -2266.97
 x_max :    1395.97
 y_min :   -2530.58
 y_max :    1771.97

Dictionary Output format:
MAP_BOUNDS_Ancient = {'x_min': -2266.971435546875, 'x_max': 1395.971